Input:  
- s3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_FILTERED_202310010000/
- s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/ ?

Output:
- s3://thesis--ec331-s3/filtered-and-melted-price-bids

In [1]:
# Compress price bids and move to the s3://thesis--ec331-s3/price-bids-compressed/ folder
import awswrangler as wr
import pandas as pd

# Base input and output paths
base_input_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/"
base_output_path = "s3://thesis--ec331-s3/price-bids-compressed/"

# List all folders (directories) in the base input path
folders = wr.s3.list_directories(base_input_path)
print(f"Found {len(folders)} folders in {base_input_path}")

for folder in folders:
    print(f"\nProcessing folder: {folder}")
    
    # List parquet files in the folder
    files = wr.s3.list_objects(folder, suffix=".parquet")
    total_files = len(files)
    
    empty_files = 0
    dataframes = []
    
    # Process each file without printing each file's details
    for file in files:
        try:
            df = wr.s3.read_parquet(file)
            if df.empty:
                empty_files += 1
            else:
                dataframes.append(df)
        except Exception as e:
            print(f"Error reading file {file}: {e}")
    
    # Combine non-empty DataFrames if any
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
    else:
        combined_df = pd.DataFrame()
    
    # Construct output folder path, preserving the original folder structure
    output_folder = folder.replace(base_input_path, base_output_path)
    # Ensure output folder path ends with '/' and define the output file name
    output_path = output_folder.rstrip("/") + "/combined.parquet"
    
    # Write the combined DataFrame to the new S3 location
    wr.s3.to_parquet(df=combined_df, path=output_path, index=False)
    
    # Print summary for the folder
    print(f"Folder Summary: {empty_files}/{total_files} files were empty.")
    print(f"Combined DataFrame with {len(combined_df)} rows saved to: {output_path}")

Found 7 folders in s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/

Processing folder: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_PUBLIC_ARCHIVE#BIDDAYOFFER#FILE01#202408010000.parquet/
Folder Summary: 0/25 files were empty.
Combined DataFrame with 304341 rows saved to: s3://thesis--ec331-s3/price-bids-compressed/RAISE1SEC_PUBLIC_ARCHIVE#BIDDAYOFFER#FILE01#202408010000.parquet/combined.parquet

Processing folder: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/
Folder Summary: 2/16 files were empty.
Combined DataFrame with 103162 rows saved to: s3://thesis--ec331-s3/price-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/combined.parquet

Processing folder: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202311010000.parquet/
Folder Summary: 5/18 files were empty.
Combined DataFrame with 149010 rows saved to: s3://thesis--ec331-s3/price-bids-compressed/RAISE1SEC_PU

In [4]:
import awswrangler as wr
import pandas as pd
import gc
import logging
import os

logging.basicConfig(level=logging.INFO)

def process_folder(input_folder: str, output_base: str) -> None:
    logging.info(f"Processing folder: {input_folder}")
    
    # List all Parquet files in the folder
    files = wr.s3.list_objects(input_folder, suffix=".parquet")
    total_files = len(files)
    logging.info(f"Found {total_files} Parquet files.")
    
    empty_files = 0
    df_list = []
    
    # Read each file (preserving all columns)
    for file in files:
        try:
            df = wr.s3.read_parquet(file)
            if df.empty:
                empty_files += 1
            else:
                df_list.append(df)
        except Exception as e:
            logging.error(f"Error reading file {file}: {e}")
    
    if not df_list:
        logging.info("No non-empty files found in this folder; skipping.")
        return
    
    # Combine DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    logging.info(f"Combined DataFrame has {len(combined_df):,} rows before deduplication.")
    
    # Convert columns to datetime (if not already)
    combined_df["SETTLEMENTDATE"] = pd.to_datetime(combined_df["SETTLEMENTDATE"], errors="coerce")
    combined_df["OFFERDATE"] = pd.to_datetime(combined_df["OFFERDATE"], errors="coerce")
    
    # Sort so that within each (DUID, SETTLEMENTDATE) group, the row with the latest OFFERDATE is last
    combined_df.sort_values(by=["DUID", "SETTLEMENTDATE", "OFFERDATE"], ascending=True, inplace=True)
    
    # Deduplicate based on [DUID, SETTLEMENTDATE], keeping the last (latest OFFERDATE)
    before_count = len(combined_df)
    deduped_df = combined_df.drop_duplicates(subset=["DUID", "SETTLEMENTDATE"], keep="last")
    after_count = len(deduped_df)
    
    logging.info(f"Deduplication: {before_count - after_count} duplicates removed ({after_count} rows remain).")
    logging.info(f"Empty files in folder: {empty_files}/{total_files}")
    
    # Construct output folder path, preserving the original structure
    # (e.g., s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_PUBLIC_... -> s3://thesis--ec331-s3/de-duped-price-bids/RAISE1SEC_PUBLIC_...)
    output_folder = input_folder.replace("s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/", output_base)
    output_path = os.path.join(output_folder.rstrip("/"), "deduped.parquet")
    
    # Write deduplicated DataFrame to the new S3 location
    wr.s3.to_parquet(df=deduped_df, path=output_path, index=False, dataset=False)
    logging.info(f"Deduped data written to: {output_path}")

def main():
    base_input_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/"
    base_output_path = "s3://thesis--ec331-s3/de-duped-price-bids/"
    
    # List all folders in the base input path
    folders = wr.s3.list_directories(base_input_path)
    logging.info(f"Found {len(folders)} folders in {base_input_path}")
    
    # Process each folder individually to manage memory usage
    for folder in folders:
        process_folder(folder, base_output_path)
        gc.collect()

if __name__ == "__main__":
    main()

INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:root:Found 7 folders in s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/
INFO:root:Processing folder: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_PUBLIC_ARCHIVE#BIDDAYOFFER#FILE01#202408010000.parquet/
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:root:Found 25 Parquet files.
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.credentials:Found credentials from IAM Role: thesis
INFO:botocore.crede

In [ ]:
import awswrangler as wr
import pandas as pd

# S3 path to your specific Parquet file
file_path = "s3://thesis--ec331-s3/de-duped-price-bids/0b2e3b04a6f846c5a51119c90cb099cc.snappy.parquet"

# Read the file into a DataFrame
df = wr.s3.read_parquet(path=file_path)

# Show the first few rows
print("\n--- HEAD OF DATAFRAME ---")
print(df.head(10))

# If your date column is named SETTLEMENTDATE, convert to datetime
# (You can skip this if it’s already a datetime)
if "SETTLEMENTDATE" in df.columns:
    df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"], errors="coerce")

# Check earliest and latest date in SETTLEMENTDATE
if "SETTLEMENTDATE" in df.columns:
    earliest = df["SETTLEMENTDATE"].min()
    latest = df["SETTLEMENTDATE"].max()
    print(f"\nEarliest SETTLEMENTDATE: {earliest}")
    print(f"Latest SETTLEMENTDATE:  {latest}")
else:
    print("\nNo SETTLEMENTDATE column found.")

print("\n--- DONE ---")

In [ ]:
unique_duids = df["DUID"].unique()         # array of unique DUIDs
len(unique_duids)